In [ ]:
# W8 Day 4 - Runtime 无状态执行
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

# 🧱 LangChat 心智模型｜第8周-Day4：Runtime 无状态执行

> **链路第四步：执行计划怎么跑起来的？**
> Runtime 是手术室不是病房——所有信息从 FrozenExecutionContext 带进来，所有结果通过 ExecutionResult 带出去。

## 📅 学习进度

```
W1  ████████████████████ ✅ Transformer与大模型训练
W2  ████████████████████ ✅ 微调与RLHF
W3  ████████████████████ ✅ RAG与知识增强
W4  ████████████████████ ✅ 推理与思维链
W5  ████████████████████ ✅ Agent与工具使用
W6  ████████████████████ ✅ LLM Agent实战
W7  ████████████████████ ✅ 数字员工架构深化
W8  ████████████████░░░░ 🔥 LangChat 心智模型 (Day4/7)
W9  ░░░░░░░░░░░░░░░░░░░░ 📐 Domain Deep Dive
W10 ░░░░░░░░░░░░░░░░░░░░ 📋 横切关注点
W11 ░░░░░░░░░░░░░░░░░░░░ 🔍 Code Reality
```

**进度: 8/13 周 (61.5%) | Day 32/49（W8阶段）**

# 🔄 往期回顾（W8 链路前三天）

| Day | 主题 | Today's Question | 与今天的关系 |
|-----|------|------------------|-------------|
| D1 | 用户意图 | 为什么 LangChat 不是 Agent Host？ | Agent Host 的请求最终要到达 Runtime |
| D2 | ApplicationContract | 为什么 Contract 不是 API 文档？ | Contract 版本作为 DeploymentRevision 闭包字段 |
| D3 | Blueprint→Compiler→IR | 为什么 Blueprint 不能直接运行？ | Compiler 产出的 IR 被 Runtime 装载执行 |

## 🎯 Today's Question

**为什么 Runtime 不保存状态？**

直觉上，"运行时"应该记住会话、缓存结果、维护上下文。
LangChat 的设计恰好相反：Runtime 是无状态的"手术室"——
所有信息从 FrozenExecutionContext 带进来，执行完通过 ExecutionResult 带出去。

# 📚 Part 1：为什么需要无状态 Runtime？

## 生活类比：手术室 vs 病房

| 角色 | 医院 | LangChat |
|------|------|----------|
| 病房 | 患者长期住院、记录病史 | 数据库 / KB / Channel 子系统 |
| 手术室 | 所有信息从病历带入，术后结果写回病历 | Runtime Layer |
| 病历 | 患者身份、病史、用药——手术室不保管 | FrozenExecutionContext |
| 手术器械包 | 无菌封装、版本号 | SkillRelease / DeploymentRevision |

## Runtime "三不原则"

| 原则 | 含义 | 来源 |
|------|------|------|
| 不读 mutable name | 不读 Blueprint/Channel/Catalog/latest | HC-4, AS §13.4 |
| 不存状态 | 所有信息从 FEC 获取，结果通过 ExecutionResult 返回 | Charter 01 §6 |
| 不做价值判断 | 不评估 Release、不调 LLM 做 Planning、不放宽 Policy | HC-2, AS §18.3-14 |

# 📚 Part 2：ADR/架构如何设计

## Single Canonical Execution Path（Charter 01 §6.2）

所有执行经过唯一入口：`execute(deployment_revision, frozen_context, input)`

- 接收 DeploymentRevision **对象**，不是 bare digest
- 传 bare digest 会抛 `BareDigestRejectedError`

## FrozenExecutionContext 不可变性（HC-1）

| 字段类别 | 包含什么 | 为什么需要 |
|----------|----------|-----------|
| identity | tenant、workspace、caller、delegation_chain | 知道"谁"在执行 |
| policy | effect_policy、depth_limit、scope_constraints | 知道"允许做什么" |
| contract_route | contract_version、deployment_id、traffic_policy | 知道"执行哪个版本" |
| artifact_digests | skill_release_digest、runtime_abi_version | 知道"运行哪个制品" |
| execution_boundary | max_duration、max_cost、provider_boundary | 安全边界 |

**HC-1: Frozen 后不可修改。** 任何变更必须生成新 FEC。

## Runtime 封闭性（WP-10a）

> `runtime/__init__.py` 文档：This package MUST NOT import `release_channel`, `catalog`, or `workflow`.

所有外部依赖通过 `execute()` 参数注入：`runtime_factory`、`kb_search_fn`、`llm_chat_fn`、`tool_call_fn`。

# 📚 Part 3：当前代码如何实现

## Runtime 包结构

```
runtime/
├── __init__.py              # 包入口 + 封闭性声明
├── canonical_entry.py       # execute() 唯一入口
├── deployment_revision.py   # 16字段闭包 + SHA-256 digest
├── frozen_execution_context.py  # Evaluation FEC
├── production.py            # Production FEC
├── loader.py                # RuntimeLoader（WP-05 stub）
├── materialize.py           # 实例化器
├── skill_bindings.py        # skill_id → workflow 映射（过渡层）
├── traffic_policy.py        # 确定性 cohort hash
├── evaluation_only_guard.py # evaluation_only 守卫
├── errors.py                # 错误类型层级
└── types.py                 # 类型别名
```

## execute() 执行流程

```
execute(deployment_revision, frozen_context, input, *, runtime_factory, ...)
  │
  ├─ 1. 类型检查：必须是 DeploymentRevision 对象
  ├─ 2. 锚定校验：FEC.digest == Revision.digest
  ├─ 3. 查找 SkillBinding
  │    └─ 找不到 → fallback ExecutionResult
  ├─ 4. 运行 workflow → 流式收集 → parse 七字段
  └─ 5. 返回 ExecutionResult
```

**关键：execute() 永不抛异常！** 所有失败返回 fallback 七字段结果。

## DeploymentRevision 16 字段闭包

`deployment_revision_digest` = SHA-256 over canonical JSON of 16 closure fields

`source_channel` 和 `evaluation_only` 不参与 digest（AS §11.3）。

## SkillBinding 过渡层

当前 9 个硬编码绑定（w01-w09），桥接 v1 template 和 v2 制品链。未来被 OCI manifest 替代。

In [ ]:
# 可视化：Runtime 无状态执行流程图
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(-1, 15)
ax.set_ylim(-1, 11)
ax.axis('off')
ax.set_title('LangChat Runtime 无状态执行流程', fontsize=18, fontweight='bold', pad=20)

# 颜色定义
c_input = '#4CAF50'
c_runtime = '#2196F3'
c_output = '#FF9800'
c_fallback = '#F44336'
c_inject = '#9C27B0'

# 输入区
ax.add_patch(mpatches.FancyBboxPatch((-0.5, 7.5), 4, 2.5, boxstyle="round,pad=0.3", facecolor=c_input, alpha=0.2, edgecolor=c_input, linewidth=2))
ax.text(1.5, 9.2, '调用者（Gateway）', ha='center', fontsize=11, fontweight='bold', color=c_input)
ax.text(1.5, 8.5, '• DeploymentRevision 对象\n• FrozenExecutionContext\n• input_payload', ha='center', fontsize=9, va='center')

# 注入区
ax.add_patch(mpatches.FancyBboxPatch((-0.5, 5), 4, 2, boxstyle="round,pad=0.3", facecolor=c_inject, alpha=0.15, edgecolor=c_inject, linewidth=2, linestyle='--'))
ax.text(1.5, 6.2, '依赖注入（参数）', ha='center', fontsize=10, fontweight='bold', color=c_inject)
ax.text(1.5, 5.5, 'runtime_factory / kb_search_fn\nllm_chat_fn / tool_call_fn', ha='center', fontsize=8, va='center', color=c_inject)

# Runtime 核心
ax.add_patch(mpatches.FancyBboxPatch((5, 4), 5.5, 5.5, boxstyle="round,pad=0.4", facecolor=c_runtime, alpha=0.1, edgecolor=c_runtime, linewidth=3))
ax.text(7.75, 8.8, '🔒 Runtime Layer（无状态）', ha='center', fontsize=13, fontweight='bold', color=c_runtime)

# Runtime 内部步骤
steps = [
    ('① 类型检查', '必须是 DeploymentRevision 对象'),
    ('② 锚定校验', 'FEC.digest == Revision.digest'),
    ('③ 查找 SkillBinding', 'skill_id → workflow template'),
    ('④ 运行 Workflow', '注入依赖 → 流式收集'),
    ('⑤ 返回结果', 'ExecutionResult 七字段'),
]
for i, (step, desc) in enumerate(steps):
    y = 8.0 - i * 0.75
    ax.text(6.2, y, step, fontsize=10, fontweight='bold', color=c_runtime)
    ax.text(9.8, y, desc, fontsize=8, ha='right', color='#555')

# 箭头：输入 → Runtime
ax.annotate('', xy=(5, 8.5), xytext=(3.7, 8.5),
            arrowprops=dict(arrowstyle='->', color=c_input, lw=2.5))
ax.annotate('', xy=(5, 6), xytext=(3.7, 6),
            arrowprops=dict(arrowstyle='->', color=c_inject, lw=2, linestyle='--'))

# 输出
ax.add_patch(mpatches.FancyBboxPatch((11.5, 6), 3.5, 3, boxstyle="round,pad=0.3", facecolor=c_output, alpha=0.15, edgecolor=c_output, linewidth=2))
ax.text(13.25, 8.3, 'ExecutionResult', ha='center', fontsize=11, fontweight='bold', color=c_output)
ax.text(13.25, 7.3, '• execution_id\n• trace_id\n• output (7字段)\n• latency_ms', ha='center', fontsize=9, va='center')

ax.annotate('', xy=(11.3, 7.5), xytext=(10.7, 7.5),
            arrowprops=dict(arrowstyle='->', color=c_output, lw=2.5))

# Fallback 路径
ax.add_patch(mpatches.FancyBboxPatch((11.5, 3), 3.5, 2, boxstyle="round,pad=0.3", facecolor=c_fallback, alpha=0.12, edgecolor=c_fallback, linewidth=2))
ax.text(13.25, 4.3, 'Fallback 结果', ha='center', fontsize=10, fontweight='bold', color=c_fallback)
ax.text(13.25, 3.5, 'skill未绑定 / workflow失败\n→ 七字段结构化降级\n→ 敏感词转人工', ha='center', fontsize=8, va='center', color=c_fallback)

ax.annotate('', xy=(11.3, 4), xytext=(10.5, 5.5),
            arrowprops=dict(arrowstyle='->', color=c_fallback, lw=1.5, linestyle=':'))
ax.text(11.2, 5.2, '失败/未找到', fontsize=7, color=c_fallback, ha='center')

# 底部注释
ax.text(7.5, 0.5, '⚡ Runtime 不保存任何状态：不缓存、不记忆、不维护会话\n所有信息通过参数传入，所有结果通过返回值传出',
        ha='center', fontsize=10, style='italic', color='#666',
        bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.8))

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第8周/w8d4_runtime_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存")

In [ ]:
# 可视化：DeploymentRevision 闭包结构与 Digest 计算
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图：16字段闭包
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')
ax1.set_title('DeploymentRevision 16字段闭包', fontsize=14, fontweight='bold')

closure_fields = [
    'skill_release_digest',
    'application_contract_version',
    'runtime_abi_version',
    'runtime_profile',
    'manifest_schema_version',
    'execution_plan_ir_schema_version',
    'frozen_context_schema_version',
    'required_artifact_media_types',
    'knowledge_snapshot_digests',
    'capability_release_digests',
    'policy_bundle_digest',
    'prompt_artifacts',
    'model_artifacts',
    'runtime_artifacts',
    'environment',
    'binding_manifest_digest',
]

for i, field in enumerate(closure_fields):
    row = i // 2
    col = i % 2
    y = 9.2 - row * 0.55
    x = 0.3 + col * 4.8
    color = '#E3F2FD' if field.endswith('digest') or field.endswith('version') else '#FFF3E0'
    ax1.add_patch(mpatches.FancyBboxPatch((x, y - 0.2), 4.3, 0.4, boxstyle="round,pad=0.1", facecolor=color, edgecolor='#999', linewidth=0.5))
    ax1.text(x + 0.15, y, f'{i+1}. {field}', fontsize=7.5, va='center', fontfamily='monospace')

ax1.text(5, 0.3, '↓ SHA-256 over canonical JSON (sorted keys, compact) ↓', ha='center', fontsize=8, color='#666')

# 右图：Digest 计算排除项
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Digest 计算包含什么 / 排除什么', fontsize=14, fontweight='bold')

# 包含
ax2.add_patch(mpatches.FancyBboxPatch((0.3, 5.5), 4.2, 4, boxstyle="round,pad=0.3", facecolor='#E8F5E9', edgecolor='#4CAF50', linewidth=2))
ax2.text(2.4, 9.1, '✅ 参与 digest', ha='center', fontsize=11, fontweight='bold', color='#2E7D32')
ax2.text(2.4, 8.3, '• 16 个闭包字段全部参与\n• canonical JSON (sorted keys)\n• compact separators\n• UTF-8 编码',
         ha='center', fontsize=9, va='center')

# 排除
ax2.add_patch(mpatches.FancyBboxPatch((5.2, 5.5), 4.5, 4, boxstyle="round,pad=0.3", facecolor='#FFEBEE', edgecolor='#F44336', linewidth=2))
ax2.text(7.45, 9.1, '❌ 不参与 digest', ha='center', fontsize=11, fontweight='bold', color='#C62828')
ax2.text(7.45, 8.3, '• source_channel（溯源信息）\n• evaluation_only（部署策略）\n• revision_id（实例标识）\n• build_run_id（构建标识）\n• operator 身份\n• 构建时间戳',
         ha='center', fontsize=9, va='center')

# 底部：两种 FEC
fec_data = [
    ('Evaluation FEC', 'v1-evaluation', 'evaluation_only=True\n构建评估/测试', '#FFF9C4', '#F9A825'),
    ('Production FEC', 'v1-production', 'evaluation_only=False\n生产业务执行', '#E1BEE7', '#7B1FA2'),
]
for i, (name, ver, desc, bg, ec) in enumerate(fec_data):
    x = 0.5 + i * 4.8
    ax2.add_patch(mpatches.FancyBboxPatch((x, 1.5), 4, 3, boxstyle="round,pad=0.3", facecolor=bg, edgecolor=ec, linewidth=2))
    ax2.text(x + 2, 3.8, name, ha='center', fontsize=10, fontweight='bold', color=ec)
    ax2.text(x + 2, 3.2, f'schema: {ver}', ha='center', fontsize=8, fontfamily='monospace', color=ec)
    ax2.text(x + 2, 2.3, desc, ha='center', fontsize=8, va='center')

ax2.text(5, 0.5, '两种 Profile 共享相同字段结构，但 evaluation_only 标志不同', ha='center', fontsize=8, color='#666')

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第8周/w8d4_closure_digest.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存")

# 📚 Part 4：Gap Analysis

| 维度 | 当前态 | 目标态 | Gap |
|------|--------|--------|-----|
| execute() canonical 入口 | ✅ 已实现 | 与目标一致 | 🟢 |
| DeploymentRevision 闭包 | ✅ 16字段完整 | 与目标一致 | 🟢 |
| Runtime 无状态 | ✅ 纯参数注入 | 与目标一致 | 🟢 |
| Runtime 封闭性 | ✅ 零 workflow import | 与目标一致 | 🟢 |
| RuntimeLoader | Stub（直接返回） | OCI pull + 验证 + Compat Matrix | 🔴 |
| Compat Matrix 三点校验 | 未实现 | Build/Deploy/Load（HC-12） | 🔴 |
| SkillBinding | 9个硬编码 | OCI manifest 驱动 | 🔴 |
| Signature 验签 | 未实现 | Sigstore cosign 两点验签 | 🔴 |
| v1 wire 适配层 | 未实现 | D-7 薄适配层 | 🟡 |

# 💡 今天多理解了什么

**以前以为：** Runtime 就是"跑代码的引擎"——加载代码、执行、返回结果，中间维护一些会话状态。

**现在知道：**

1. Runtime 是"手术室"不是"病房"——不保存任何状态
2. `execute()` 接收 DeploymentRevision **对象**，不是 bare digest——阻止 Runtime 自己去 Registry 拉取
3. FrozenExecutionContext 是"封印的病历"——HC-1 不可修改，HC-2 只能收紧 Policy
4. Runtime 包是"密封盒子"——零 workflow import，执行框架通过参数注入
5. `execute()` **永不抛异常**——所有失败返回 fallback 七字段结果
6. SkillBinding 是过渡层——9 个硬编码绑定最终被 OCI manifest 替代

# 🔮 重新设计还会这样做吗？

**会。每一个设计决策都保留。**

- 无状态 → 水平扩展的前提
- FEC 不可变 → 审计的基础
- 封闭性 → 可替换执行框架
- 不抛异常 → 优秀的 API 契约
- Sensitive keyword fallback → 安全设计而非 hack

# 📝 Daily Engineering Log

| 类型 | 内容 |
|------|------|
| **新增** | Runtime Layer 完整结构：execute() / DeploymentRevision / FEC / SkillBinding |
| **新增** | Runtime 三不原则：不读 mutable name / 不存状态 / 不做价值判断 |
| **新增** | execute() fallback 机制：永不抛异常，敏感词转人工 |
| **新增** | TrafficPolicy 确定性 cohort hash：SHA-256 → mod 100 |
| **新增** | Runtime 包封闭性（WP-10a）：零 workflow import |
| **修改** | 以前认为 Runtime 是"跑代码的引擎"；现在知道是"无状态手术室" |
| **确认** | source_channel 和 evaluation_only 不参与 digest 计算 |
| **遗留** | RuntimeLoader 是 stub，无真实 OCI pull |
| **遗留** | SkillBinding 过渡层待 OCI manifest 替代 |
| **技术债** | Compatibility Matrix 三点校验未实现 |
| **技术债** | Signature pre-load 验签未实现 |
| **下一步** | Day5: Capability + Connector → Enterprise System |

# 📖 术语表

| 英文 | 音标 | 中文 |
|------|------|------|
| Runtime | /ˈraɪntʌɪm/ | 运行时——无状态执行层 |
| DeploymentRevision | /dɪˈplɔɪmənt rɪˈvɪʒn/ | 部署修订——16字段不可变闭包 |
| FrozenExecutionContext | /ˈfroʊzən ɪksˈkjuːʃn ˈkɒntekst/ | 冻结执行上下文 |
| RuntimeABI | /ˈraɪntʌɪm ˌeɪ biː ˈaɪ/ | 运行时二进制接口版本契约 |
| Compatibility Matrix | /kəmˌpætəˈbɪləti ˈmeɪtrɪks/ | 兼容性矩阵 |
| TrafficPolicy | /ˈtræfɪk ˈpɒləsi/ | 流量策略——灰度路由 |
| Cohort | /ˈkoʊhɔːrt/ | 队列——0-99百分位桶 |
| SkillBinding | /skɪl ˈbaɪndɪŋ/ | 技能绑定（过渡层） |
| Canonical Entry | /kəˈnɒnɪkl ˈentri/ | 规范入口 execute() |
| Bare Digest | /ber ˈdaɪdʒest/ | 裸摘要——被 execute() 拒绝 |
| Hermetic | /hɜːrˈmetɪk/ | 密封的——零外部 import |
| ExecutionResult | /ˌeksɪˈkjuːʃn rɪˈzʌlt/ | 执行结果——含七字段 output |

# ❓ 课堂练习

1. Runtime 可以修改 FEC 中的 Policy 吗？ → ___
2. execute() 接受 bare digest 字符串吗？ → ___
3. Runtime 包可以 import workflow 模块吗？ → ___
4. TrafficPolicy 允许 revision_id="latest" 吗？ → ___
5. workflow 失败时 execute() 会抛异常吗？ → ___

## 课后测试

**Q1:** 为什么 Runtime 不保存状态？
- A) 数据库不支持 → B) 保存状态破坏水平扩展 → C) 用户不需要 → D) FEC 已有所有信息

**Q2:** Runtime 包"封闭性"意味着什么？
- A) 沙箱容器 → B) 不 import workflow/channel/catalog，依赖参数注入 → C) 不暴露API → D) 只能用Rust

**Q3:** SkillBinding 找不到时 execute() 怎么做？
- A) 抛异常 → B) HTTP 404 → C) 返回 fallback ExecutionResult → D) 创建新绑定

# 📚 真实参考

| 来源 | 路径/章节 |
|------|-----------|
| Charter 01 §6.2 | Single Canonical Execution Path |
| Charter 01 §8 | Runtime Compatibility Matrix |
| Artifact Spec §13 | Runtime Layer 行为规范 |
| Artifact Spec §13.4 | HC-4 不读 mutable name |
| Artifact Spec §14 | FrozenExecutionContext 契约 |
| ADR-007 §3.2 | HC-1~HC-16 约束 |
| ADR-007 §7 | D-4 FEC wire 表示 |
| ADR-007 §10 | D-7 canonical 端点演进 |
| 代码 | `runtime/canonical_entry.py` — execute() 唯一入口 |
| 代码 | `runtime/deployment_revision.py` — 16字段闭包 |
| 代码 | `runtime/frozen_execution_context.py` — Evaluation FEC |
| 代码 | `runtime/production.py` — Production FEC |
| 代码 | `runtime/skill_bindings.py` — 过渡层映射 |
| 代码 | `runtime/traffic_policy.py` — cohort hash 路由 |
| 代码 | `runtime/loader.py` — RuntimeLoader stub |
| 代码 | `runtime/materialize.py` — 实例化器 |
| 代码 | `runtime/types.py` — 注入类型别名 |